In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df=pd.read_excel(r"C:\Users\SALMAN PC\Downloads\online+retail\Online Retail.xlsx")

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.shape


(541909, 8)

In [5]:
# for i in df.columns:
#     # print(df[i].value_counts())
#     print(df[i].unique())
#     print("="*75)

In [6]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(5268)

In [8]:
# (1452/541909)*100
(135080/541909)*100

24.926694334288598

In [9]:
df["StockCode"].unique()

array(['85123A', 71053, '84406B', ..., '90214U', '47591b', 23843],
      shape=(4070,), dtype=object)

In [10]:
df.shape

(541909, 8)

In [11]:
df=df.drop("CustomerID",axis=1)

In [12]:
# market=[]
# for i in range(0,df.shape[0]):
#     cus=[]
#     for j in df.columns:
#         if type(df[j][i])==str:
#         cus.append(df[j][i])
#  market.append(cus)

In [13]:
df.shape

(541909, 7)

In [14]:
df=df.drop_duplicates()

In [15]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,United Kingdom


In [16]:
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df = df.dropna(subset=['Description'])

In [17]:
basket = df.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().fillna(0)


In [18]:
# basket = basket.apply(lambda x: 1 if x > 0 else 0)
# basket = (basket > 0).astype(int)
basket_encoded = basket.map(lambda x: 1 if x > 0 else 0)
basket_bool = basket_encoded.astype(bool)

In [19]:
print(f"Matrix shape : {basket_bool.shape}  (invoices × products)")
print("\nSample (5 rows × 5 cols):")
print(basket_bool.iloc[:5, :5])

Matrix shape : (19960, 4026)  (invoices × products)

Sample (5 rows × 5 cols):
Description  4 PURPLE FLOCK DINNER CANDLES  50'S CHRISTMAS GIFT BAG LARGE  \
InvoiceNo                                                                   
536365                               False                          False   
536366                               False                          False   
536367                               False                          False   
536368                               False                          False   
536369                               False                          False   

Description  DOLLY GIRL BEAKER  I LOVE LONDON MINI BACKPACK  \
InvoiceNo                                                     
536365                   False                        False   
536366                   False                        False   
536367                   False                        False   
536368                   False                        False   
536

In [20]:
from mlxtend.frequent_patterns import apriori

In [21]:
frequent_itemsets=apriori(basket_bool,min_support=0.04,use_colnames=True,max_len=3).sort_values(by=["support"])

In [22]:
frequent_itemsets

,support,itemsets
41,0.040230,frozenset({POPCORN HOLDER})
38,0.041082,frozenset({PAPER CHAIN KIT VINTAGE CHRISTMAS})
21,0.041182,frozenset({JUMBO BAG STRAWBERRY})
65,0.041333,"frozenset({JUMBO BAG PINK POLKADOT, JUMBO BAG ..."
1,0.041483,frozenset({60 TEATIME FAIRY CAKE CASES})
...,...,...
30,0.078357,frozenset({LUNCH BAG RED RETROSPOT})
39,0.084419,frozenset({PARTY BUNTING})
47,0.099599,frozenset({REGENCY CAKESTAND 3 TIER})
20,0.104659,frozenset({JUMBO BAG RED RETROSPOT})


In [23]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values("lift", ascending=False)
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))

                            antecedents                           consequents  \
0  frozenset({JUMBO BAG PINK POLKADOT})  frozenset({JUMBO BAG RED RETROSPOT})   
1  frozenset({JUMBO BAG RED RETROSPOT})  frozenset({JUMBO BAG PINK POLKADOT})   

    support  confidence      lift  
0  0.041333    0.677340  6.471855  
1  0.041333    0.394926  6.471855  
